Question 15.2

In the videos, we saw the “diet problem”. (The diet problem is one of the first large-scale optimization
problems to be studied in practice. Back in the 1930’s and 40’s, the Army wanted to meet the nutritional
requirements of its soldiers while minimizing the cost.) In this homework you get to solve a diet problem
with real data. The data is given in the file diet.xls.
1. Formulate an optimization model (a linear program) to find the cheapest diet that satisfies the
maximum and minimum daily nutrition constraints, and solve it using PuLP. Turn in your code
and the solution. (The optimal solution should be a diet of air-popped popcorn, poached eggs,
oranges, raw iceberg lettuce, raw celery, and frozen broccoli. UGH!)
2. Please add to your model the following constraints (which might require adding more variables)
and solve the new model:
a. If a food is selected, then a minimum of 1/10 serving must be chosen. (Hint: now you will
need two variables for each food i: whether it is chosen, and how much is part of the diet.
You’ll also need to write a constraint to link them.)
b. Many people dislike celery and frozen broccoli. So at most one, but not both, can be selected.
c. To get day-to-day variety in protein, at least 3 kinds of meat/poultry/fish/eggs must be
selected. [If something is ambiguous (e.g., should bean-and-bacon soup be considered
meat?), just call it whatever you think is appropriate – I want you to learn how to write this
type of constraint, but I don’t really care whether we agree on how to classify foods!]
If you want to see what a more full-sized problem would look like, try solving your models for the file
diet_large.xls, which is a low-cholesterol diet model (rather than minimizing cost, the goal is to minimize
cholesterol intake). I don’t know anyone who’d want to eat this diet – the optimal solution includes dried
chrysanthemum garland, raw beluga whale flipper, freeze-dried parsley, etc. – which shows why it’s
necessary to add additional constraints beyond the basic ones we saw in the video! [Note: there are
many optimal solutions, all with zero cholesterol, so you might get a different one.
It probably won’t be much more appetizing than mine.]


In [1]:
import pandas as pd
import pulp
from datetime import datetime

In [2]:
df = pd.read_csv('diet.csv')
display(df)

,Foods,Price/ Serving,Serving Size,Calories,Cholesterol mg,Total_Fat g,Sodium mg,Carbohydrates g,Dietary_Fiber g,Protein g,Vit_A IU,Vit_C IU,Calcium mg,Iron mg
0,Frozen Broccoli,$0.16,10 Oz Pkg,73.8,0.0,0.8,68.2,13.6,8.5,8.0,5867.4,160.2,159.0,2.3
1,"Carrots,Raw",$0.07,1/2 Cup Shredded,23.7,0.0,0.1,19.2,5.6,1.6,0.6,15471.0,5.1,14.9,0.3
2,"Celery, Raw",$0.04,1 Stalk,6.4,0.0,0.1,34.8,1.5,0.7,0.3,53.6,2.8,16.0,0.2
3,Frozen Corn,$0.18,1/2 Cup,72.2,0.0,0.6,2.5,17.1,2.0,2.5,106.6,5.2,3.3,0.3
4,"Lettuce,Iceberg,Raw",$0.02,1 Leaf,2.6,0.0,0.0,1.8,0.4,0.3,0.2,66.0,0.8,3.8,0.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,"Crm Mshrm Soup,W/Mlk",$0.65,1 C (8 Fl Oz),203.4,19.8,13.6,1076.3,15.0,0.5,6.1,153.8,2.2,178.6,0.6
63,"Beanbacn Soup,W/Watr",$0.67,1 C (8 Fl Oz),172.0,2.5,5.9,951.3,22.8,8.6,7.9,888.0,1.5,81.0,2.0
64,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
65,NaN,NaN,Minimum daily intake,1500.0,30.0,20.0,800.0,130.0,125.0,60.0,1000.0,400.0,700.0,10.0


In [10]:
# Load your CSV file
df = pd.read_csv("diet.csv")
df['Price/ Serving'] = df['Price/ Serving'].replace('[\$,]', '', regex=True).astype(float)
df.rename(columns={'Foods': 'Food', 'Price/ Serving': 'Cost'}, inplace=True)

# Split food and nutrient constraints
food_df = df.iloc[:64].copy()
min_row = df[df['Serving Size'] == 'Minimum daily intake'].iloc[0]
max_row = df[df['Serving Size'] == 'Maximum daily intake'].iloc[0]

# Build data structures
foods = food_df['Food'].tolist()
cost = dict(zip(food_df['Food'], food_df['Cost']))
nutrients = df.columns.difference(['Food', 'Cost', 'Serving Size']).tolist()
nutrient_data = {n: dict(zip(food_df['Food'], food_df[n])) for n in nutrients}
min_req = {n: float(min_row[n]) for n in nutrients}
max_req = {n: float(max_row[n]) for n in nutrients}

# Optimization model
prob = pulp.LpProblem("Diet_Problem", pulp.LpMinimize)
x = pulp.LpVariable.dicts("x", foods, lowBound=0, cat="Continuous")
y = pulp.LpVariable.dicts("y", foods, cat="Binary")

# Objective
prob += pulp.lpSum(cost[f] * x[f] for f in foods), "Total_Cost"

# Nutrient constraints
for nutrient in nutrients:
    prob += pulp.lpSum(nutrient_data[nutrient][f] * x[f] for f in foods) >= min_req[nutrient]
    prob += pulp.lpSum(nutrient_data[nutrient][f] * x[f] for f in foods) <= max_req[nutrient]
    
prob.solve()

# Print results
if pulp.LpStatus[prob.status] == 'Optimal':
    print("Optimal solution found!\n")
    total_cost = pulp.value(prob.objective)
    for f in foods:
        if x[f].varValue > 0:
            print(f"{f}: {x[f].varValue:.2f} servings")
    print(f"\nTotal Cost: ${total_cost:.2f}")
else:
    print("No optimal solution found.")

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/joeshin/miniconda3/lib/python3.10/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/vr/l0fpjt7d5hjcx1kh48ktf2ym0000gn/T/f29250bbfa754fa89b5d2726b2bff55e-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/vr/l0fpjt7d5hjcx1kh48ktf2ym0000gn/T/f29250bbfa754fa89b5d2726b2bff55e-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 27 COLUMNS
At line 1286 RHS
At line 1309 BOUNDS
At line 1310 ENDATA
Problem MODEL has 22 rows, 64 columns and 1194 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 22 (0) rows, 64 (0) columns and 1194 (0) elements
0  Obj 0 Primal inf 21.63092 (11)
9  Obj 4.3371168
Optimal - objective value 4.3371168
Optimal objective 4.33711681 - 9 iterations time 0.002
Option for printingOptions changed from normal to all
Total time (CPU seconds):       0.00   (W

In [11]:
# Binary linkage
M = 100
epsilon = 0.1
for f in foods:
    prob += x[f] >= epsilon * y[f]
    prob += x[f] <= M * y[f]

# Celery/Broccoli mutual exclusion
prob += y["Celery, Raw"] + y["Frozen Broccoli"] <= 1

# At least 3 meat-related foods
meat_keywords = ["beef", "chicken", "pork", "turkey", "fish", "egg", "tuna", "ham", "salmon", "sardines", "frankfurter", "bacon"]
meat_foods = [f for f in foods if any(k in f.lower() for k in meat_keywords)]
prob += pulp.lpSum([y[f] for f in meat_foods]) >= 3

# Solve
prob.solve()

# Results
if pulp.LpStatus[prob.status] == 'Optimal':
    result_df = pd.DataFrame({
        'Food': [f for f in foods if x[f].varValue > 0],
        'Servings': [round(x[f].varValue, 2) for f in foods if x[f].varValue > 0],
        'Cost per Serving': [cost[f] for f in foods if x[f].varValue > 0]
    })
    result_df['Total Cost'] = result_df['Servings'] * result_df['Cost per Serving']
    print("Optimal Diet Plan:")
    print(result_df)
    print("\nTotal Cost: $", round(pulp.value(prob.objective), 2))
else:
    print("No optimal solution found.")

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/joeshin/miniconda3/lib/python3.10/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/vr/l0fpjt7d5hjcx1kh48ktf2ym0000gn/T/72697333157d4ec99f5241faad7f5975-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /var/folders/vr/l0fpjt7d5hjcx1kh48ktf2ym0000gn/T/72697333157d4ec99f5241faad7f5975-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 157 COLUMNS
At line 1814 RHS
At line 1967 BOUNDS
At line 2032 ENDATA
Problem MODEL has 152 rows, 128 columns and 1464 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 4.38082 - 0.00 seconds
Cgl0003I 0 fixed, 0 tightened bounds, 62 strengthened rows, 0 substitutions
Cgl0004I processed model has 141 rows, 128 columns (64 integer (64 of which binary)) and 867 elements
Cbc0038I Initial state - 6 integers unsatisfied sum - 1.04669


In [4]:
df = pd.read_csv('school_lunch.csv')
display(df)

FileNotFoundError: [Errno 2] No such file or directory: 'school_lunch.csv'

In [ ]:
food_df = df.iloc[:-2].copy()

min_row = df.iloc[-2]

max_row = df.iloc[-1]

foods = food_df['Food'].tolist()

cost = dict(zip(food_df['Food'], food_df['Cost']))

nutrients = [col for col in df.columns if col not in ['Food', 'Cost']]

nutrient_data = {n: dict(zip(food_df['Food'], food_df[n])) for n in nutrients}

min_req = {n: float(min_row[n]) for n in nutrients}
max_req = {n: float(max_row[n]) for n in nutrients}

# Step 1. Initialize the LP object
prob = pulp.LpProblem("School_Lunch_Optimization", pulp.LpMinimize)

# Step 2. Create continuous decision variables – can also use list
x = pulp.LpVariable.dicts("x", foods, cat="Continuous", lowBound=0)
# Example: x = {"pizza": x_pizza, "nuggets": x_nuggets, ...}

# Optional – one way to create binary variables, then link to continuous variables
y = {
    "Pizza": pulp.LpVariable("y_Pizza", cat="Binary"),
    "Nuggets": pulp.LpVariable("y_Nuggets", cat="Binary")
}

# Step 3. Objective function
prob += pulp.lpSum(cost[i] * x[i] for i in foods), "Total_Cost"

# Step 4. Add constraints
# Each nutrient has a min and max requirement
for nutrient in nutrients:
    prob += pulp.lpSum(nutrient_data[nutrient][i] * x[i] for i in foods) >= min_req[nutrient], f"Min_{nutrient}"  # min constraint
    prob += pulp.lpSum(nutrient_data[nutrient][i] * x[i] for i in foods) <= max_req[nutrient], f"Max_{nutrient}"  # max constraint

# Link binary to continuous
M = 1000   # Big M method
epsilon = 0.0001  # Small positive number to enforce minimum if selected



# Constraint: can't have both pizza and nuggets
prob += y["Pizza"] + y["Nuggets"] <= 1, "cannot_have_both_pizza_and_nugget"

# Link binary and continuous for Pizza
prob += x["Pizza"] <= M * y["Pizza"], "With Big M Pizza"
prob += x["Pizza"] >= epsilon * y["Pizza"], "With epsilon Pizza"

# Link binary and continuous for Nuggets
prob += x["Nuggets"] <= M * y["Nuggets"], "With Big M Nuggets"
prob += x["Nuggets"] >= epsilon * y["Nuggets"], "With epsilon Nuggets"

# Step 5. Solve
prob.solve()

# Initialize placeholder for result
result = {}



if pulp.LpStatus[prob.status] == 'Optimal':
    result['status'] = 'Optimal'
    result['total_cost'] = round(pulp.value(prob.objective), 2)
    result['servings'] = {f"x_{i}": round(x[i].varValue, 2) for i in foods}
else:
    result['status'] = pulp.LpStatus[prob.status]
    result['message'] = "No optimal solution found."